In [ ]:
library(tidyverse)
library(nycflights13)

In [ ]:
library(lubridate)
flights |> 
  select(year,month,day,time_hour) |> 
  mutate(
    dep_hour=hour(time_hour),
    flight_date=make_datetime(year,month,day,dep_hour),
    .keep="none"
  ) |> tail()

In [ ]:
#rowwise modifications
df <- tibble(
  student = c("Alex", "Blair"),
  q1 = c(1, 2),
  q2 = c(2, 3),
  q3 = c(4, 5)
)
df

df |> 
  rowwise() |> 
  mutate(total_marks =sum(c(q1,q2,q3)))


In [ ]:
df %>%
  group_by(group) %>%
  arrange(value, .by_group = TRUE)

In [ ]:
flights |> 
  group_by(carrier) |> 
  mutate(avg_delay=mean(arr_delay,na.rm=TRUE),.keep="used")

#.by parameter
flights |> 
  mutate(avg_delay=mean(arr_delay,na.rm=TRUE),.by = "carrier",.keep="used")
#no need ungroup()

In [ ]:
#pick() vs rowmeans()
flights |> 
  mutate(
    avg_delay=rowMeans(pick(dep_delay,arr_delay),na.rm=TRUE),
    .keep="used"
  ) |>  
  mutate(avg_depDelay=mean(c(dep_delay,arr_delay,avg_delay),na.rm=TRUE))
#in mutate , by default it takes the columwise operation 
#for rowwise() , you need tp explicity provide rowwise() / rowMeans+pick()

In [ ]:
#if your goal top 3 delays per carrier.
flights |> 
  slice_max(arr_delay,n=3,by=carrier)

In [ ]:
flights |> 
  group_by(carrier) |> 
  arrange(desc(arr_delay)) |> slice_head(n=3)

In [ ]:
flights |> 
  arrange(carrier,desc(arr_delay)) |> 
  mutate(
    row_id=row_number(),
    .by=carrier
  ) |> filter(row_id <=3)

In [ ]:
#mutate --> keep the same rows
#reframe() --> can return multiple rows per group it combines head(sort)
flights |> 
  reframe(
    top_delay=head(sort(arr_delay,decreasing=TRUE),3),
    .by=carrier
  )

In [ ]:
#remove a column
flights |> 
  mutate(
    gain=arr_delay-dep_delay,
    .keep="used",
    gain=NULL #its remvoes the column
  )

# A tibble: 336,776 × 2
   dep_delay arr_delay
       <dbl>     <dbl>
 1         2        11
 2         4        20
 3         2        33
 4        -1       -18
 5        -6       -25
 6        -4        12
 7        -5        19
 8        -3       -14
 9        -3        -8
10        -2         8
# ℹ 336,766 more rows
# ℹ Use `print(n = ...)` to see more rows

In [ ]:
#transumte - its keep only new column
flights |> 
  transmute(
    gain=arr_delay-dep_delay
  )

# A tibble: 336,776 × 1
    gain
   <dbl>
 1     9
 2    16
 3    31
 4   -17
 5   -19
 6    16
 7    24
 8   -11
 9    -5
10    10
# ℹ 336,766 more rows
# ℹ Use `print(n = ...)` to see more rows

In [59]:
#Custom function 
#custom function + mutate()
# dep_delay <=0 , on_time,  <=30 = minor_delay, NA= Unknown, other major_delay

delay_category <- function(delay){
  case_when(
    is.na(delay)~"unknown",
    delay<=0~"on_time",
    delay<=30~"minor_delay",
    TRUE~"major_delay"
  )
}

flights |> 
  mutate(dep_status=delay_category(dep_delay),.keep="used")

# A tibble: 336,776 × 2
   dep_delay dep_status 
       <dbl> <chr>      
 1         2 minor_delay
 2         4 minor_delay
 3         2 minor_delay
 4        -1 on_time    
 5        -6 on_time    
 6        -4 on_time    
 7        -5 on_time    
 8        -3 on_time    
 9        -3 on_time    
10        -2 on_time    
# ℹ 336,766 more rows
# ℹ Use `print(n = ...)` to see more rows

In [ ]:
#function with multiple inputs
calc_speed <- function(dist,air_time){
  (dist/air_time)*60
}

flights |> 
  mutate(speed=calc_speed(distance,air_time),.keep="used")

# A tibble: 336,776 × 3
   air_time distance speed
      <dbl>    <dbl> <dbl>
 1      227     1400  370.
 2      227     1416  374.
 3      160     1089  408.
 4      183     1576  517.
 5      116      762  394.
 6      150      719  288.
 7      158     1065  404.
 8       53      229  259.
 9      140      944  405.
10      138      733  319.
# ℹ 336,766 more rows
# ℹ Use `print(n = ...)` to see more rows